In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Configurează atenuarea erorilor

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** Versiunea beta a unui nou model de execuție este acum disponibilă. Modelul de execuție direcționat oferă mai multă flexibilitate la personalizarea fluxului tău de lucru pentru atenuarea erorilor. Consultă ghidul [Modelul de execuție direcționat](/guides/directed-execution-model) pentru mai multe informații.


<details>
<summary><b>Versiuni pachete</b></summary>

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm utilizarea acestor versiuni sau a unora mai noi.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Tehnicile de atenuare a erorilor le permit utilizatorilor să diminueze erorile din Circuit prin
modelarea zgomotului dispozitivului la momentul execuției. Aceasta produce de obicei
suprasarcini de pre-procesare cuantică legate de antrenarea modelului și
suprasarcini de post-procesare clasică pentru atenuarea erorilor din rezultatele brute,
folosind modelul generat.

Primitivul Estimator suportă mai multe tehnici de atenuare a erorilor, inclusiv [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation), [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne), [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec) și [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). Consultă [Tehnici de atenuare și suprimare a erorilor](error-mitigation-and-suppression-techniques) pentru o explicație a fiecăreia. Când folosești primitive, poți activa sau dezactiva metode individuale. Consultă secțiunea [Setări personalizate pentru erori](#advanced-error) pentru detalii.

> **Note:** Sampler nu suportă atenuarea erorilor, dar poți folosi pachetul [mthree](https://qiskit.github.io/qiskit-addon-mthree/) (matrix-free measurement mitigation) pentru a efectua atenuarea erorilor local.

Estimator suportă și `resilience_level`. Nivelul de reziliență specifică cât de multă reziliență să construiești împotriva
erorilor. Nivelurile mai ridicate generează rezultate mai precise, cu prețul unor
timpi de procesare mai lungi. Nivelurile de reziliență pot fi folosite pentru a configura
compromisul cost/precizie atunci când aplici atenuarea erorilor la interogarea ta primitivă. Atenuarea erorilor reduce erorile (biasul) din rezultate prin procesarea
ieșirilor dintr-o colecție, sau ansamblu, de circuite înrudite. Gradul de reducere a erorilor depinde de metoda aplicată. Nivelul de reziliență abstractizează alegerea detaliată a metodei de atenuare a erorilor pentru a le permite
utilizatorilor să raționeze despre compromisul cost/precizie adecvat
aplicației lor.

Având în vedere aceasta, fiecare nivel corespunde unei metode sau unor metode cu
niveluri crescânde de suprasarcină de eșantionare cuantică, pentru a-ți permite să experimentezi
cu diferite compromisuri timp-precizie. Tabelul următor îți arată
ce niveluri și metode corespunzătoare sunt disponibile pentru fiecare dintre
primitive.

> **Info:** Atenuarea erorilor este specifică sarcinii, deci tehnicile pe care le poți
> aplica variază în funcție de dacă eșantionezi o distribuție sau generezi
> valori de așteptare.

<span id="resilience-table"></span>

Estimator suportă următoarele niveluri de reziliență. Sampler nu suportă niveluri de reziliență.

| Nivel de reziliență | Definiție                                                                                            | Tehnică                                                                 |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0                | Fără atenuare                                                                                         | Niciuna                                                                      |
| 1 [Implicit]      | Costuri minime de atenuare: Atenuează erorile asociate cu erorile de citire               | Twirled Readout Error eXtinction (TREX) cu răsucire a măsurătorii             |
| 2                | Costuri medii de atenuare. Reduce de obicei biasul în estimatori, dar nu garantează un rezultat fără bias. | Nivelul 1 + Zero Noise Extrapolation (ZNE) și răsucire Gate

> **Info:** Nivelurile de reziliență sunt în prezent în versiune beta, astfel că suprasarcina de eșantionare și
> calitatea soluției vor varia de la un Circuit la altul. Funcții noi,
> opțiuni avansate și instrumente de gestionare vor fi lansate în mod continuu. Nu se garantează că metodele specifice de atenuare a erorilor vor fi
> aplicate la fiecare nivel de reziliență.

## Configurează Estimator cu niveluri de reziliență
Poți folosi nivelurile de reziliență pentru a specifica tehnici de atenuare a erorilor, sau poți seta tehnici personalizate individual, după cum este descris în [Setări personalizate pentru erori.](#advanced-error)

<details>
<summary>Nivelul de reziliență 0</summary>

Nicio atenuare a erorilor nu este aplicată programului utilizatorului.

</details>

<details>
<summary>Nivelul de reziliență 1</summary>

Nivelul 1 aplică **atenuarea erorilor de citire** și **răsucirea măsurătorii** prin aplicarea unei tehnici fără model, cunoscute
drept Twirled Readout Error eXtinction (TREX). Reduce eroarea de măsurare
prin diagonalizarea canalului de zgomot asociat cu măsurătoarea, prin
răsturnarea aleatorie a qubiților prin Gate-uri X imediat înainte de măsurătoare. Un
termen de redimensionare din canalul de zgomot diagonal este învățat prin
testarea circuitelor aleatorii inițializate în starea zero. Aceasta permite
serviciului să elimine biasul din valorile de așteptare care rezultă din
zgomotul de citire. Această abordare este descrisă mai detaliat în [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

<details>
<summary>Nivelul de reziliență 2</summary>

Nivelul 2 aplică **tehnicile de atenuare a erorilor incluse în nivelul 1** și, de asemenea, aplică **răsucirea Gate** și folosește **metoda Zero Noise Extrapolation (ZNE)**. ZNE calculează o
valoare de așteptare a observabilului pentru diferiți factori de zgomot
(etapa de amplificare) și apoi folosește valorile de așteptare măsurate pentru a
deduce valoarea de așteptare ideală la limita de zgomot zero (etapa de
extrapolare). Această abordare tinde să reducă erorile în valorile de așteptare, dar
nu garantează un rezultat fără bias.

![Această imagine arată un grafic. Axa x este etichetată Factorul de amplificare a zgomotului. Axa y este etichetată Valoarea de așteptare. O linie cu pantă ascendentă este etichetată Valoare atenuată. Punctele de lângă linie sunt valori amplificate prin zgomot. Există o linie orizontală puțin deasupra axei X etichetată Valoare exactă.](../docs/images/guides/configure-error-mitigation/resilience-2.svg "Ilustrare a metodei ZNE")

Suprasarcina acestei metode se scalează cu numărul de factori de zgomot. Setările
implicite eșantionează valoarea de așteptare la trei factori de zgomot,
conducând la o suprasarcină de aproximativ 3x când se folosește acest nivel de reziliență.

În Nivelul 2, metoda TREX răstoarnă aleatoriu qubiții prin Gate-uri X imediat înainte de măsurătoare
și răstoarnă bitul măsurat corespunzător dacă un Gate X a fost aplicat. Această abordare este descrisă mai detaliat în [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

### Exemplu
Interfața `EstimatorV2` le permite utilizatorilor să lucreze fără dificultăți cu varietatea de
metode de atenuare a erorilor pentru a reduce eroarea în valorile de așteptare ale
observabililor. Următorul cod folosește Zero Noise Extrapolation și atenuarea erorilor de citire prin simpla
setare a `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Setări personalizate pentru erori
Poți activa și dezactiva metode individuale de atenuare și suprimare a erorilor, inclusiv decuplarea dinamică, răsucirea Gate și a măsurătorii, atenuarea erorilor de măsurătoare, PEC și ZNE. Consultă [Tehnici de atenuare și suprimare a erorilor](error-mitigation-and-suppression-techniques) pentru o explicație a fiecăreia.

> **Note:** - Nu toate opțiunile sunt disponibile pentru ambele primitive. Consultă [tabelul opțiunilor disponibile](runtime-options-overview#options-table) pentru lista opțiunilor disponibile.
> - Nu toate metodele funcționează împreună pe toate tipurile de circuite. Consultă [tabelul de compatibilitate a funcțiilor](runtime-options-overview#options-compatibility-table) pentru detalii.

<Tabs>
  <TabItem value="Estimator" label="Estimator">
    ```python
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}")
print(f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}")
```
  </TabItem>

  <TabItem value="Sampler" label="Sampler">